# Reconstruction MAE : original vs reconstruit

On prend une fenêtre des données de pretraining (TLE SpaceTrack), on la passe dans le MAE
préentraîné avec son masquage habituel, et on trace les séries originales contre les séries
reconstruites. Les patchs masqués sont grisés : c'est là que le décodeur invente.

Ce qu'on cherche à voir : sur un patch masqué qui contient une rupture de dynamique, le
décodeur restitue-t-il le saut ou remet-il une interpolation lisse ?

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from ml.inference import load_checkpoint, reconstruct_window, load_spacetrack_features

base = Path.cwd()
device = torch.device('cpu')

CKPT_ID = '2026-08-11_14-43-36'  ## <-- le run de pretrain à regarder
ckpt_path = Path(os.path.join(base, '..', '..', 'outputs', 'ml', 'pretrain', CKPT_ID, 'checkpoints', 'best.pt'))
data_dir = Path(os.path.join(base, '..', '..', 'data', 'raw', 'spacetrack'))

mae, cfg, mean, scale = load_checkpoint(ckpt_path, device)
window_size, patch_size = cfg.data.window_size, cfg.model.patch_size
print(f"{cfg.model.name} | window {window_size} | patch {patch_size} | "
      f"{mae.num_patches} patchs | masking_ratio {mae.masking_ratio}")

In [ ]:
## Mêmes données et même normalisation qu'au pretraining
per_obj = load_spacetrack_features(data_dir, cfg.data.dataset, mean, scale)

per_obj = {norad: X for norad, X in per_obj.items() if len(X) >= window_size}
print(f"{len(per_obj)} objets d'au moins {window_size} TLE")

In [ ]:
FEATURE_COLS = ['dt', 'sma', 'k', 'h', 'p', 'q', 'cosM', 'sinM', 'sma_diff', 'q_diff', 'p_diff']
TO_PLOT = ['sma', 'sma_diff', 'k', 'h']  ## dt exclu : sa loss de reconstruction est à poids nul


def plot_reconstruction(norad, window_start, seed=0):
    X = per_obj[norad]
    x = X[window_start:window_start + window_size].T                  ## (F, W) normalisé
    recon, masked = reconstruct_window(mae, x, device=device, seed=seed)

    x_phys, recon_phys = x.T * scale + mean, recon.T * scale + mean    ## unités physiques
    t = np.arange(window_start, window_start + window_size)

    fig, axes = plt.subplots(len(TO_PLOT), 1, figsize=(14, 2.6 * len(TO_PLOT)), sharex=True)
    for ax, name in zip(axes, TO_PLOT):
        col = FEATURE_COLS.index(name)
        for p in masked:
            ax.axvspan(t[p * patch_size], t[min((p + 1) * patch_size, window_size - 1)],
                       color='grey', alpha=0.18, lw=0)
        ## On ne trace la reconstruction QUE sur les patchs masqués : la loss du MAE n'est
        ## calculée que sur ceux-là (cf. TimeSeriesMAE.forward, pred/target extraits avec
        ## ids_mask). Sur les patchs visibles, pred_space_proj n'a jamais été supervisée et
        ## sort n'importe quoi -- l'y tracer ne ferait qu'induire en erreur.
        pred = np.full(window_size, np.nan)
        for p in masked:
            sl = slice(p * patch_size, (p + 1) * patch_size)
            pred[sl] = recon_phys[sl, col]

        ax.plot(t, x_phys[:, col], color='black', lw=1.4, label='original')
        ax.plot(t, pred, color='tab:red', lw=1.6, label='reconstruit (patchs masqués)')
        ax.set_ylabel(name)
    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].set_title(f"norad {norad} — fenêtre [{window_start}, {window_start + window_size}) "
                      f"— patchs masqués grisés")
    axes[-1].set_xlabel('TimeIndex (TLE)')
    fig.tight_layout()
    plt.show()


norad = next(iter(per_obj))
plot_reconstruction(norad, window_start=0)

In [ ]:
## Quelques autres objets / fenêtres
for norad in list(per_obj)[1:4]:
    plot_reconstruction(norad, window_start=len(per_obj[norad]) // 2 - window_size // 2)

In [ ]:
## Loss de reconstruction par objet, avec la loss du pretrain (dt à poids nul)
from ml.utils import MaskedChannelMSE

w = torch.ones(len(FEATURE_COLS))
w[FEATURE_COLS.index('dt')] = 0.0
loss_fn = MaskedChannelMSE(w)


@torch.no_grad()
def object_loss(norad, n_windows=8, seed=0):
    """Loss moyenne sur n_windows fenêtres, et sa décomposition par canal."""
    X = per_obj[norad]
    starts = np.linspace(0, len(X) - window_size, n_windows).astype(int)
    xb = torch.from_numpy(np.stack([X[s:s + window_size].T for s in starts])).float()

    torch.manual_seed(seed)  ## le masquage de mae.forward est aléatoire
    pred, target = mae(xb)   ## (B, N_masqués, F*P) : le forward ne renvoie que les masqués
    per_channel = (pred - target).reshape(len(starts), -1, len(FEATURE_COLS), patch_size) \
                                 .pow(2).mean(dim=(0, 1, 3))
    return float(loss_fn(pred, target)), per_channel.numpy()


losses = {norad: object_loss(norad) for norad in per_obj}
best = sorted(losses, key=lambda n: losses[n][0])

print("les 5 objets les mieux reconstruits :")
for norad in best[:5]:
    print(f"  norad {norad}: loss {losses[norad][0]:.5f}")

print(f"\ndécomposition par canal du meilleur (norad {best[0]}) :")
for name, value in sorted(zip(FEATURE_COLS, losses[best[0]][1]), key=lambda kv: -kv[1]):
    print(f"  {name:10s} {value:.5f}")

In [ ]:
## Ce que le modèle reconstruit le mieux
for norad in best[:3]:
    plot_reconstruction(norad, window_start=len(per_obj[norad]) // 2 - window_size // 2)